# Huấn luyện Mô hình Nhánh 2 Tay (Two-Hands Branch Classifier)
Notebook này thực hiện huấn luyện mô hình cho cử chỉ 2 tay kết hợp thân trên (100 đặc trưng x 30 khung hình), đánh giá F1-Score, vẽ Confusion Matrix và xuất tệp `two_hands_classifier.tflite`.

In [1]:
import re
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

from models.sequence_classifier import convert_to_tflite

print(f"TensorFlow Version: {tf.__version__}")

ModuleNotFoundError: No module named 'models'

In [ ]:
# 1. Nạp dữ liệu từ data/two_hands/
data_dir = "data/two_hands"
X, y = [], []
label_names = []

label_csv = os.path.join(data_dir, "keypoint_classifier_label.csv")
if os.path.exists(label_csv):
    with open(label_csv, "r", encoding="utf-8") as f:
        label_names = [line.strip() for line in f if line.strip()]
else:
    label_names = ["XinChao", "CamOn", "XinLoi", "YeuThuong", "TamBiet"]

print(f"Danh sách nhãn Nhánh 2 Tay ({len(label_names)} lớp): {label_names}")

# Nạp tệp .npy nếu có (hỗ trợ cả tên nhãn gốc và tên an toàn trên Windows)
for idx, label in enumerate(label_names):
    safe_label = re.sub(r'[\\/*?:"<>|]', '-', label).strip('. ')
    folder = os.path.join(data_dir, safe_label)
    if not os.path.exists(folder):
        folder = os.path.join(data_dir, label)
    files = glob.glob(os.path.join(folder, "*.npy"))
    for fpath in files:
        arr = np.load(fpath)
        if arr.shape == (30, 100):
            X.append(arr)
            y.append(idx)

# Tạo dữ liệu giả lập nếu chưa có mẫu thực tế
if len(X) == 0:
    print("[DEMO] Chưa phát hiện dữ liệu thực tế. Đang tạo 150 mẫu giả lập để chạy thử...")
    for c in range(len(label_names)):
        for _ in range(30):
            sample = np.random.normal(c * 0.2, 0.1, (30, 100)).astype(np.float32)
            X.append(sample)
            y.append(c)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
print(f"X shape: {X.shape}, y shape: {y.shape}")


Danh sách nhãn Nhánh 2 Tay (28 lớp): ['cảm ơn', 'xin lỗi', 'aaa', 'làm ơn', 'giá', 'không', 'cái gì', 'ở đâu', 'khi nào', 'ai', 'tại sao', 'bao nhiêu', 'muốn', 'giúp', 'tạm biệt', 'tên', '1', '2', '3', '4', '5', '6', '7', '8', '9', '0', '00', '000']
X shape: (480, 30, 100), y shape: (480,)


In [ ]:
# 2. Phân chia Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Dựng mô hình Bi-LSTM cho Nhánh 2 Tay kết hợp Thân trên
input_dim = X_train.shape[2]
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, input_dim)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True, unroll=True)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, unroll=True)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(label_names), activation="softmax")
], name="TwoHandsClassifierModel")

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=16)


Epoch 1/25

24/24 [==============================] - 7s 70ms/step - loss: 1.6609 - accuracy: 0.5651 - val_loss: 0.6672 - val_accuracy: 0.6667
Epoch 2/25

24/24 [==============================] - 0s 9ms/step - loss: 0.6314 - accuracy: 0.6927 - val_loss: 0.5503 - val_accuracy: 0.7188
Epoch 3/25

24/24 [==============================] - 0s 9ms/step - loss: 0.5493 - accuracy: 0.7318 - val_loss: 0.5252 - val_accuracy: 0.7396
Epoch 4/25

24/24 [==============================] - 0s 9ms/step - loss: 0.5274 - accuracy: 0.7448 - val_loss: 0.4913 - val_accuracy: 0.7604
Epoch 5/25

24/24 [==============================] - 0s 9ms/step - loss: 0.4799 - accuracy: 0.7656 - val_loss: 0.4268 - val_accuracy: 0.8646
Epoch 6/25

24/24 [==============================] - 0s 9ms/step - loss: 0.3978 - accuracy: 0.8359 - val_loss: 0.5811 - val_accuracy: 0.7083
Epoch 7/25

24/24 [==============================] - 0s 9ms/step - loss: 0.4505 - accuracy: 0.8281 - val_loss: 0.5470 - val_accuracy: 0.7292
Epoch 8/25



In [ ]:
# 4. Đánh giá & Vẽ Ma Trận Nhầm Lẫn (Confusion Matrix)
y_pred = np.argmax(model.predict(X_test), axis=1)

# Báo cáo đánh giá chi tiết theo các nhãn có dữ liệu
present_classes = sorted(list(set(y_test) | set(y_pred)))
present_names = [label_names[i] for i in present_classes]
print(classification_report(y_test, y_pred, labels=present_classes, target_names=present_names, zero_division=0))

# Vẽ heatmap ma trận nhầm lẫn
cm = confusion_matrix(y_test, y_pred, labels=present_classes)
plt.figure(figsize=(10, 8), dpi=150)
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Purples',
    xticklabels=present_names,
    yticklabels=present_names,
    linewidths=0.5,
    linecolor='gray'
)
plt.title('Ma Trận Nhầm Lẫn (Confusion Matrix) - Nhánh 2 Tay', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Nhãn Dự Đoán (Predicted Label)', fontsize=12, labelpad=10)
plt.ylabel('Nhãn Thực Tế (True Label)', fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig('model/two_hands_classifier/confusion_matrix.png')
plt.show()

# 5. Xuất TFLite Float16 cho Nhánh 2 Tay
out_tflite = "model/two_hands_classifier/two_hands_classifier.tflite"
convert_to_tflite(model, output_path=out_tflite, float16_quantization=True)



3/3 [==============================] - 2s 4ms/step
              precision    recall  f1-score   support

      cảm ơn       1.00      1.00      1.00        16
     xin lỗi       1.00      1.00      1.00        16
      làm ơn       1.00      1.00      1.00        16
           1       1.00      1.00      1.00        16
           2       0.94      1.00      0.97        16
           3       1.00      0.94      0.97        16

    accuracy                           0.99        96
   macro avg       0.99      0.99      0.99        96
weighted avg       0.99      0.99      0.99        96

[SUCCESS] Đã xuất mô hình TFLite tại: model/two_hands_classifier/two_hands_classifier.tflite (Kích thước: 765.51 KB)
